<a href="https://www.kaggle.com/code/faihaj/creating-a-new-hope-for-darpa-optc-ecar-dataset?scriptVersionId=306448542" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
from kaggle_secrets import UserSecretsClient
import json
import os

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_API_TOKEN")

# Primary Work
Look into the paper: [A New Hope for DARPA OpTC](https://inria.hal.science/hal-05474126v1)
<br>
More details: https://correctedoptc.inria.fr/
<br>
In this notebook we will work on the ecar folder.

## Downloading the corrected dataset
You can find the corrected version of the OpTC dataset [here](https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi:10.57745/UXCWOC).
<br>
I will download each file as I have in [this notebook](https://www.kaggle.com/code/faihaj/optc-partially-corrected-dataset-creation). We will use the individual Id of each tar file.
<br>
Then as there size is huge to store, we will convert it to parquet file by using the same method as in [this notebook](https://www.kaggle.com/code/faihaj/optc-json-to-parquet-table). Furthermore, we can take full advantage of parquets columnar storage by flattening the ***properites*** column of the dataset as done so in [this notebook](https://www.kaggle.com/code/faihaj/optc-host51-sep16-exploration-with-flattening).
<br>
So I tried to use [`TarStreamETL`](https://github.com/FaIhAjAlAmToPu/tar-stream-etl.git) to create the corrected yet unlabelled version of the OpTC datasets first.
<br>
Here is the list of datasets I created using it:
* [sep 18 part 1](https://www.kaggle.com/datasets/faihaj/testing-optc-sep18-part-1)
* [sep 18 part 2](https://www.kaggle.com/datasets/faihaj/testing-optc-sep18-part-2)
* [sep 19 part 1](https://www.kaggle.com/datasets/faihaj/testing-optc-sep19-part-1)
* [sep 19 part 2](https://www.kaggle.com/datasets/faihaj/testing-optc-sep19-part-2)
* [sep 20 part 1](https://www.kaggle.com/datasets/faihaj/testing-optc-sep20-part-1)
* [sep 20 part 2](https://www.kaggle.com/datasets/faihaj/testing-optc-sep20-part-2)
* [sep 21 part 1](https://www.kaggle.com/datasets/faihaj/testing-optc-sep21-part-1)
* [sep 21 part 2](https://www.kaggle.com/datasets/faihaj/testing-optc-sep21-part-2)
* [sep 22 part 1](https://www.kaggle.com/datasets/faihaj/testing-optc-sep22-part-1)
* [sep 22 part 2](https://www.kaggle.com/datasets/faihaj/testing-optc-sep22-part-2)

### Creating the sep 25 dataset
let's create the sep 25 dataset using `tar-strem-etl` package.

In [ ]:
!pip install tar-stream-etl

In [ ]:
day_id = {16:"EG2WAI",
          17:"LS5EKZ",
          18:"E5C3H6",
          19:"YOU7U5",
          20:"CSQGOH",
          21:"OFJJMW",
          22:"0D2RNJ",
          23:"9LTJMR",
          24:"KFNPJ7",
          25:"DNWHUD"
         }

In [ ]:
day = 25
url = f"https://entrepot.recherche.data.gouv.fr/api/access/datafile/:persistentId?persistentId=doi:10.57745/{day_id[day]}"

### kaggle dataset
it requires the metadata as `json`

In [ ]:
metadata = {
  "title": f"testing OpTC-ecar {day}th",
  "id": f"{os.environ['KAGGLE_USERNAME']}/testing-optc-sep{day}", 
  "licenses": [{"name": "MIT"}]
}

In [ ]:
# from tar_stream_etl import TarStreamETL
# TarStreamETL(url=url, download_dir = "/kaggle/working/dwnld", metadata=metadata).run()

## Disk size exceeding problem
All the previous approaches has problem with either disk size or producer stream error due to the extra time given to threads flushing the data into dataset thus receiving timeout from the tar server.
<br>
So, I flush it to huggingface dataset and then try again if we gets error. Next runtime will be skipping the already downloaded files from downloading into the disk thus wasting time but saving space.
<br>
Here are the link to the notebook that was used to create the datasets:
* [sep 16](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=306099793)
* [sep 17](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305802417)
* [sep 18](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305773535)
* [sep 19](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=306072567)
* [sep 20](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305898343)
* [sep 21](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305911031)
* [sep 22](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305892720)
* [sep 23](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305892816)
* [sep 24](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305959482)
* [sep 25](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305933113)

## Fixing Errors
As can be seen for the case of [sep 21](https://www.kaggle.com/code/faihaj/fast-forward-tarstreametl-for-huggingface-for-optc?scriptVersionId=305911031) dataset creation, there was an error while processing some `json files`. But we had to skip those files as there are no contents there as that can be evident [here](https://www.kaggle.com/code/faihaj/finding-faulty-files-of-optc-transformation?scriptVersionId=305958732).

## Labelled dataset Links
The labelled versions for the partially corrected DARPA OpTC datasets can be found in huggingface.
* [ecar-sep16](https://huggingface.co/datasets/equalNull/labelled-optc-sep16)
* [ecar-sep17](https://huggingface.co/datasets/equalNull/labelled-optc-sep17)
* [ecar-sep18](https://huggingface.co/datasets/equalNull/labelled-optc-sep18)
* [ecar-sep19](https://huggingface.co/datasets/equalNull/labelled-optc-sep19)
* [ecar-sep20](https://huggingface.co/datasets/equalNull/labelled-optc-sep20)
* [ecar-sep21](https://huggingface.co/datasets/equalNull/labelled-optc-sep21)
* [ecar-sep22](https://huggingface.co/datasets/equalNull/labelled-optc-sep22)
* [ecar-sep23](https://huggingface.co/datasets/equalNull/labelled-optc-sep23)
* [ecar-sep24](https://huggingface.co/datasets/equalNull/labelled-optc-sep24)
* [ecar-sep25](https://huggingface.co/datasets/equalNull/labelled-optc-sep25)